# YouTube → Uzbek subtitles (BuzzASR/uzbek)
Select **Runtime → Change runtime type → GPU**, edit the settings, then **Run all**.
The notebook downloads audio, converts it to 16 kHz mono, runs the Uzbek checkpoint with native sequential long-form timestamps, and downloads a UTF-8 SRT.

A public URL is tried **without authentication first**. YouTube can block Colab's IP, require login, or require extra playback tokens; no notebook can guarantee every URL. If download fails, upload the matching audio/video or set a runtime audio path. Use media you are authorized to access.

Cookies are optional credentials—not a guaranteed fix. See the cookie instructions below. No cookies belong in notebook code, outputs, or Drive.


In [1]:
#@title 1. Settings
SOURCE = "youtube" #@param ["youtube", "upload", "path"]
YOUTUBE_URL = "https://youtu.be/RDpD0b" #@param {type:"string"}
AUDIO_PATH = "" #@param {type:"string"}
USE_COOKIE_SECRET = False #@param {type:"boolean"}
UPLOAD_IF_DOWNLOAD_FAILS = False #@param {type:"boolean"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
# Optional: Colab's key icon → secret YOUTUBE_COOKIES → grant this notebook access.
# The secret VALUE must be the complete YouTube-only Netscape cookies.txt content.
COOKIE_SECRET_NAME = "YOUTUBE_COOKIES"
MODEL_ID = "BuzzASR/uzbek"


## Optional cookies and the no-cookie fallback
Colab cannot read the Chrome/Safari profile on your computer. Automatic Google Colab login is not YouTube authentication for yt-dlp.

For optional authentication, follow [yt-dlp's YouTube-only export instructions](https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies): use a separate private browser session, export only `youtube.com` cookies in Netscape format, then close that session as instructed. Avoid an all-sites cookie export. In Colab's **Secrets** (key icon), create `YOUTUBE_COOKIES`, paste that file's contents as the value, grant this notebook access, and enable `USE_COOKIE_SECRET`. Never paste credentials into a code cell or chat. Only grant access to a trusted notebook; its code can read granted secrets. Using account cookies carries account/blocking risk.

The notebook reads the secret only after anonymous download fails, filters it to YouTube domains, creates a private temporary file, and deletes it after the download attempt (also on normal interruption/error). This still sends credentials to Google's runtime; it is not zero-risk. Expired cookies require a fresh export. Hard runtime termination relies on deletion of the ephemeral runtime, not the Python cleanup handler.

To keep cookies entirely local, download the audio on your own computer and choose `SOURCE="upload"`; or put an existing file in Colab and choose `SOURCE="path"`. Do not trim audio: its start time and edits must match the video. For large files, using an existing runtime/Drive file path is preferable to browser upload.

### Programmatic download on your computer (cookies stay local)
With [uv](https://docs.astral.sh/uv/getting-started/installation/) and [Node ≥22](https://nodejs.org/en/download) installed, run the following locally—not inside Colab. Replace the URL and browser name if needed. Your OS may ask permission to read the browser's encrypted cookies.
```sh
uvx --from 'yt-dlp[default]' yt-dlp --js-runtimes node --cookies-from-browser chrome --no-playlist -f 'bestaudio/best' -o '%(id)s.%(ext)s' 'https://youtu.be/aKMRivL09mY'
```
Upload the resulting audio to the notebook. This avoids exporting a cookie file to the cloud; YouTube can still reject the local download.

References: [cookies FAQ](https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp), [JS runtime requirements](https://github.com/yt-dlp/yt-dlp/wiki/EJS), [Colab Secrets API](https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py).


# Tested environment
- Colab runtime: `2026.07`
- GPU: G4; T4; A100
- Python: <version>
- PyTorch / CUDA: <versions>
- Transformers: <version>

Other environments have not been validated. Use them at your own risk;
compatibility, performance, and results may differ.

In [ ]:
#@title 2. Install tools and check the GPU (no model download yet)
import sys, subprocess, shutil, platform, re, hashlib, tarfile, tempfile
from pathlib import Path
from urllib.request import urlopen
from importlib.metadata import version

if "transformers" in sys.modules and version("transformers") != "5.17.0":
    raise RuntimeError("Start a fresh Colab session before running this notebook.")
print("Installing dependencies...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers==5.17.0", "accelerate", "yt-dlp[default]", "tqdm"],
               check=True)
# Keep Colab's preinstalled CUDA-compatible PyTorch; do not replace it.
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

def ensure_node():
    existing = shutil.which("node")
    if existing:
        reported = subprocess.check_output([existing, "--version"], text=True).strip()
        if int(reported.lstrip("v").split(".")[0]) >= 22:
            return existing
    if platform.system() != "Linux" or platform.machine() not in ("x86_64", "amd64"):
        raise RuntimeError("This installer expects a Linux x86_64 Colab runtime.")
    # Download an official Node 22 binary and check its published SHA-256.
    base = "https://nodejs.org/dist/latest-v22.x/"
    with urlopen(base + "SHASUMS256.txt", timeout=60) as response:
        checksums = response.read().decode("utf-8")
    matches = re.findall(r"^([0-9a-f]{64})\s+(node-v22\.\d+\.\d+-linux-x64\.tar\.xz)$",
                         checksums, flags=re.MULTILINE)
    if len(matches) != 1:
        raise RuntimeError("Could not identify the official Node 22 Linux archive.")
    digest, name = matches[0]
    target = Path("/content/uzbek_subtitle_tools")
    target.mkdir(exist_ok=True)
    node = target / "node"
    with tempfile.TemporaryDirectory(prefix="node-install-") as folder:
        archive = Path(folder) / name
        with urlopen(base + name, timeout=60) as response, archive.open("wb") as out:
            shutil.copyfileobj(response, out)
        if hashlib.sha256(archive.read_bytes()).hexdigest() != digest:
            raise RuntimeError("Node archive checksum failed.")
        member = name.removesuffix(".tar.xz") + "/bin/node"
        # Extract only the known binary, not arbitrary archive paths.
        with tarfile.open(archive, "r:xz") as bundle:
            info = bundle.getmember(member)
            if not info.isfile():
                raise RuntimeError("Unexpected Node archive member.")
            with bundle.extractfile(info) as src, node.open("wb") as out:
                shutil.copyfileobj(src, out)
    node.chmod(0o755)
    return str(node)

NODE = ensure_node() if SOURCE == "youtube" else None
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime and rerun; this notebook requires CUDA.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| Transformers:", version("transformers"),
      "| yt-dlp:", version("yt-dlp"))
if NODE:
    print("YouTube JS runtime:", subprocess.check_output([NODE, "--version"], text=True).strip())


In [ ]:
#@title 3. Define download, credential, and subtitle helpers
import contextlib, http.cookiejar, json, math, os, textwrap, wave, warnings
from urllib.parse import urlparse, parse_qs

def youtube_url(url):
    parsed = urlparse(url.strip())
    if parsed.scheme not in ("http", "https") or parsed.username or parsed.password:
        raise ValueError("Provide an http(s) YouTube video URL.")
    host = (parsed.hostname or "").lower()
    parts = parsed.path.strip("/").split("/")
    if host in ("youtu.be", "www.youtu.be"):
        video_id = parts[0]
    elif host in ("youtube.com", "www.youtube.com", "m.youtube.com", "music.youtube.com"):
        video_id = (parse_qs(parsed.query).get("v", [""])[0] if parsed.path == "/watch"
                    else parts[1] if len(parts) == 2 and parts[0] in ("shorts", "embed", "live")
                    else "")
    else:
        raise ValueError("Only YouTube video URLs are accepted.")
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise ValueError("A single video URL is required, not a playlist/channel URL.")
    # Drop playlist, tracking and start-time parameters: subtitles start at video time zero.
    return "https://www.youtube.com/watch?v=" + video_id, video_id

@contextlib.contextmanager
def private_youtube_cookiefile(cookie_text):
    # No raw cookie text or parsing errors are printed.
    with tempfile.TemporaryDirectory(prefix="youtube-auth-") as folder:
        filename = Path(folder) / "cookies.txt"
        fd = os.open(filename, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
        with os.fdopen(fd, "w", encoding="utf-8", newline="\n") as stream:
            stream.write(cookie_text.replace("\r\n", "\n").replace("\r", "\n"))
        cookie_text = None
        valid = False
        try:
            jar = http.cookiejar.MozillaCookieJar(str(filename))
            with warnings.catch_warnings():
                # CookieJar can otherwise emit malformed cookie contents in a warning.
                warnings.simplefilter("ignore")
                jar.load(ignore_discard=True, ignore_expires=True)
            for cookie in list(jar):
                domain = cookie.domain.lstrip(".").lower()
                if not (domain == "youtube.com" or domain.endswith(".youtube.com")):
                    jar.clear(cookie.domain, cookie.path, cookie.name)
            if list(jar):
                jar.save(ignore_discard=True, ignore_expires=True)
                filename.chmod(0o600)
                valid = True
        except (http.cookiejar.LoadError, ValueError, OSError):
            pass
        if not valid:
            raise ValueError("Secret must contain a valid Netscape cookie file with YouTube cookies.")
        yield filename

class SafeDownloadLog:
    def __init__(self):
        self.reasons = set()
    def debug(self, message):
        pass
    def info(self, message):
        pass
    def warning(self, message):
        self._classify(message)
    def error(self, message):
        self._classify(message)
    def _classify(self, message):
        value = str(message).lower()
        for needle, label in (
            ("not a bot", "YouTube bot confirmation"),
            ("sign in", "YouTube login requirement"),
            ("403", "HTTP 403 (access/IP/playback restriction)"),
            ("po token", "YouTube playback-token requirement"),
            ("javascript", "YouTube JavaScript challenge"),
            ("private video", "Private/unavailable video"),
            ("not available", "Unavailable media/format"),
        ):
            if needle in value:
                self.reasons.add(label)

def download_audio(url, folder, node, cookiefile=None):
    from yt_dlp import YoutubeDL
    from yt_dlp.utils import DownloadError
    logger = SafeDownloadLog()
    last_percent = [-10]
    def hook(status):
        if status.get("status") == "downloading":
            total = status.get("total_bytes") or status.get("total_bytes_estimate")
            if total:
                percent = min(100, int(100 * status.get("downloaded_bytes", 0) / total))
                if percent >= last_percent[0] + 10:
                    print(f"Audio download: {percent}%", flush=True)
                    last_percent[0] = percent
        elif status.get("status") == "finished":
            print("Audio download finished.", flush=True)
    options = {
        "format": "bestaudio/best", "noplaylist": True,
        "outtmpl": str(Path(folder) / "audio.%(ext)s"),
        "js_runtimes": {"node": {"path": str(node)}},
        "quiet": True, "no_warnings": True, "logger": logger,
        "progress_hooks": [hook], "socket_timeout": 30,
        "retries": 2, "fragment_retries": 2, "cachedir": False,
        # Refuse currently live streams instead of downloading indefinitely.
        "match_filter": lambda info, *, incomplete=False:
            "Live streams are unsupported; use a finished video."
            if info.get("is_live") else None,
    }
    if cookiefile is not None:
        options["cookiefile"] = str(cookiefile)
    failed = False
    try:
        with YoutubeDL(options) as ydl:
            info = ydl.extract_info(url, download=True)
            if not info:
                logger.reasons.add("Live/unavailable media")
                failed = True
            else:
                saved = Path(ydl.prepare_filename(info))
                if not saved.is_file() or saved.stat().st_size == 0:
                    failed = True
    except DownloadError:
        failed = True
    if failed:
        reason = "; ".join(sorted(logger.reasons)) or "Download/network failure"
        # Raise outside except: avoid exposing raw yt-dlp URLs or credential diagnostics.
        raise RuntimeError(reason + ". Use matching local audio if downloading remains blocked.")
    return saved

def upload_audio(folder):
    from google.colab import files
    print("Select exactly one matching audio/video file (not a cookie file).", flush=True)
    uploaded = files.upload(target_dir=str(folder))
    paths = list(uploaded)
    uploaded.clear()  # Release the upload's in-memory byte copies.
    if len(paths) != 1:
        raise ValueError("Upload exactly one audio/video file, then rerun this cell.")
    path = Path(paths[0])
    if not path.is_file():
        raise ValueError("The uploaded file was not saved.")
    return path

def srt_time(seconds):
    milliseconds = round(seconds * 1000)
    hours, rest = divmod(milliseconds, 3_600_000)
    minutes, rest = divmod(rest, 60_000)
    seconds, milliseconds = divmod(rest, 1000)
    return f"{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}"

def make_srt(segments, duration):
    cues, review = [], {"overlap": 0, "many_lines": 0, "long_cue": 0}
    previous_start, previous_end = -1, -1
    for segment in segments:
        text = " ".join(segment["text"].split())
        if not text:
            continue
        start, end = float(segment["start"]), float(segment["end"])
        if not (math.isfinite(start) and math.isfinite(end)):
            raise ValueError("Invalid model timestamps; review the saved segments JSON.")
        start, end = max(0.0, start), min(duration, end)
        start_ms, end_ms = round(start * 1000), round(end * 1000)
        if start_ms < previous_start or end_ms <= start_ms:
            raise ValueError("Invalid/out-of-order caption interval; review the saved segments JSON.")
        if start_ms < previous_end:
            review["overlap"] += 1
        lines = textwrap.wrap(text, width=42, break_long_words=False, break_on_hyphens=False)
        review["many_lines"] += int(len(lines) > 2)
        review["long_cue"] += int(end - start > 8)
        body = "\n".join(lines)
        cues.append(f"{len(cues) + 1}\n{srt_time(start)} --> {srt_time(end)}\n{body}\n")
        previous_start, previous_end = start_ms, end_ms
    if not cues:
        raise ValueError("No captions generated; review the saved segments JSON.")
    return "\n".join(cues) + "\n", review


In [ ]:
#@title 4. Get audio (anonymous first, optional credentials, upload fallback)
import numpy as np
ROOT = Path("/content/uzbek_subtitles")
ROOT.mkdir(exist_ok=True)
JOB = Path(tempfile.mkdtemp(prefix="run-", dir=ROOT))
label = "interview"
audio_file = None

if SOURCE == "youtube":
    url, label = youtube_url(YOUTUBE_URL)
    print("Trying anonymous YouTube audio download...", flush=True)
    try:
        audio_file = download_audio(url, JOB, NODE)
    except RuntimeError as error:
        print(str(error))  # Sanitized reason only.
    if audio_file is None and USE_COOKIE_SECRET:
        print("Trying the explicitly enabled Colab cookie secret...", flush=True)
        def authenticated_download():
            from google.colab import userdata
            secret = None
            try:
                secret = userdata.get(COOKIE_SECRET_NAME)
                with private_youtube_cookiefile(secret) as cookiefile:
                    return download_audio(url, JOB, NODE, cookiefile)
            finally:
                secret = None
        try:
            audio_file = authenticated_download()
        except Exception:
            # Do not print the exception: malformed files may contain credentials.
            print("Authenticated download failed. Check Secret/access/export; "
                  "YouTube may still block this runtime. No cookie details were printed.")
    if audio_file is None:
        if not UPLOAD_IF_DOWNLOAD_FAILS:
            raise RuntimeError("Download blocked. Choose SOURCE='upload' or SOURCE='path'.")
        print("Switching to local audio upload; no model has been loaded yet.")
        audio_file = upload_audio(JOB)
elif SOURCE == "upload":
    audio_file = upload_audio(JOB)
elif SOURCE == "path":
    audio_file = Path(AUDIO_PATH).expanduser()
    if not audio_file.is_file():
        raise ValueError("AUDIO_PATH must point to an existing file in this runtime.")
else:
    raise ValueError("Invalid SOURCE setting.")

print("Converting audio to 16 kHz mono...", flush=True)
normalized = JOB / "normalized.wav"
conversion = subprocess.run(
    ["ffmpeg", "-nostdin", "-v", "error", "-i", str(audio_file),
     "-map", "0:a:0", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(normalized)],
    capture_output=True, text=True)
if conversion.returncode:
    raise RuntimeError("FFmpeg could not decode this audio/video. Check that the file is valid.")
with wave.open(str(normalized), "rb") as stream:
    if (stream.getnchannels(), stream.getframerate(), stream.getsampwidth()) != (1, 16000, 2):
        raise RuntimeError("Unexpected normalized audio format.")
    audio = np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").astype(np.float32)
audio /= 32768.0
duration = audio.size / 16000
if duration == 0:
    raise ValueError("The input contains no audio.")
print(f"Audio ready: {duration / 60:.1f} minutes. Job folder: {JOB}", flush=True)


## Transcription and timing
This uses **BuzzASR/uzbek**, not a replacement generic Whisper checkpoint. Native long-form decoding handles windows sequentially and retains cross-window context; it is not independent chunk batching. The progress bar measures audio frames completed, not generated tokens, and can pause while a window decodes.

Baseline decoding uses one beam and no extra repeat penalties to preserve genuine repeated speech; these choices are not a guarantee of best accuracy. Timestamps and text must be checked against the video, especially near silences and window boundaries. SRT line wrapping preserves segment timings; long segments are flagged rather than split with invented word timings. [Whisper long-form API](https://huggingface.co/docs/transformers/v5.17.0/en/model_doc/whisper) · [Uzbek checkpoint](https://huggingface.co/BuzzASR/uzbek).


In [ ]:
#@title 5. Load the Uzbek model on the GPU
from transformers import WhisperForConditionalGeneration, WhisperProcessor
print("Loading BuzzASR/uzbek...", flush=True)
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
print("Model ready:", model.device, model.dtype, flush=True)


In [ ]:
#@title 6. Transcribe with audio progress/ETA; save timed text immediately
from tqdm.auto import tqdm
import time

print("Preparing full-length audio features (no truncation)...", flush=True)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt",
                   truncation=False, padding="longest", return_attention_mask=True)
feats = inputs.input_features.to(device=model.device, dtype=model.dtype)
mask = inputs.attention_mask.to(model.device)
del inputs
print("Generating timed transcription...", flush=True)
started = time.monotonic()
with tqdm(total=feats.shape[-1], desc="Transcribing", unit="frame") as bar:
    def monitor(p):
        done, total = p[0].detach().cpu().tolist()  # A single audio file.
        bar.total = int(total)
        bar.update(max(0, min(int(done), int(total)) - bar.n))
    with torch.inference_mode():
        result = model.generate(
            feats, attention_mask=mask,
            return_timestamps=True, return_segments=True,
            # Uzbek language/task prompt comes from this checkpoint's generation config.
            num_beams=1, do_sample=False,
            no_repeat_ngram_size=0, repetition_penalty=1.0,
            monitor_progress=monitor,
        )
    bar.update(max(0, bar.total - bar.n))
segments = [
    {"start": float(segment["start"]), "end": float(segment["end"]),
     "text": processor.decode(
         segment["tokens"].detach().cpu() if torch.is_tensor(segment["tokens"])
         else segment["tokens"], skip_special_tokens=True).strip()}
    for segment in result["segments"][0]
]
segments_path = JOB / f"{label}.segments.json"
with segments_path.open("x", encoding="utf-8") as stream:
    json.dump({"model": MODEL_ID, "duration": duration, "segments": segments},
              stream, ensure_ascii=False, indent=2)
del feats, mask, result
torch.cuda.empty_cache()
elapsed = time.monotonic() - started
print(f"Transcription finished in {elapsed / 60:.1f} min "
      f"({duration / elapsed:.1f}× audio realtime).")
print("Timed transcript saved:", segments_path)


In [ ]:
#@title 7. Validate and download the SRT (rerun this cell without retranscribing)
from google.colab import files
# Read the saved artifact so export can be retried or edited independently.
with segments_path.open(encoding="utf-8") as stream:
    saved = json.load(stream)
captions, review = make_srt(saved["segments"], saved["duration"])
srt_path = segments_path.with_name(segments_path.name.removesuffix(".segments.json") + ".srt")
# Only replace this run's derived SRT; preserve its original timed transcript.
srt_path.write_text(captions, encoding="utf-8")
print("Saved:", srt_path)
if any(review.values()):
    print("Review flags (counts):", review)
print("Preview:\n" + captions[:700])
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/Uzbek_subtitles") / JOB.name
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(srt_path, destination / srt_path.name)
    shutil.copy2(segments_path, destination / segments_path.name)
    print("SRT and timed text backed up:", destination)
print("Check text/timing. For a video you manage: YouTube Studio → Subtitles → Upload file → With timing.")
files.download(str(srt_path))


## If something fails
- **YouTube bot/login/403/PO-token errors:** installing JavaScript fixes missing runtime support, not all access restrictions. Try the optional secret once, or use matching local audio. Cookie refresh is not fully automatic.
- **Large local files:** use the Files sidebar/Drive and `SOURCE="path"`. Drive files require mounting Drive before the audio cell. Optional output backup occurs only at export; runtime reset before transcription ends loses unfinished work.
- **Invalid subtitle timestamps:** the timed text is already in `*.segments.json`. Correct that artifact, then rerun only the export cell; do not rerun transcription just to fix formatting.
- **Long/overlapping cues:** review flags are not an accuracy score. The notebook does not promise word-level alignment or broadcast-style two-line captions.
- **Interrupted transcription:** progress is observable, but inference has no incremental/resumable checkpoint. Keep the runtime active until completion.
- **Model/runtime compatibility:** use a fresh Colab GPU runtime. Transformers is pinned to the version shown in your screenshots; yt-dlp updates at setup because YouTube changes frequently.

Stop inference with Colab's square stop button. Disconnect/delete the runtime when finished to stop paid compute. Delete any local cookie export when no longer needed.
